# BreaKHis 80:10:10 — Google Colab GPU edition

This notebook keeps the original four-model training and evaluation workflow, changes the patient-level split to 80%/10%/10%, uses Colab GPU and local runtime storage for faster image reads, and saves checkpoints to Google Drive as `vit_best.pth`, `swin_best.pth`, `deit_best.pth`, and `beit_best.pth` under `MyDrive/Breast_Cancer_Research/breakhis_savedmodels/breakhis_80_10_10/`.

**Before running:** choose Runtime → Change runtime type → GPU. For Kaggle download, place `kaggle.json` in the root of Google Drive or upload it when prompted. The dataset is downloaded to `/content` for speed; model weights and evaluation plots are saved to Drive.


In [ ]:
# Google Colab setup
# Before running: Runtime > Change runtime type > select a GPU (T4/L4/A100 if available).
# PyTorch is preinstalled in Colab; install only the remaining dependencies.
!pip -q install timm kaggle scikit-learn matplotlib pandas pillow opencv-python-headless tqdm


In [ ]:
import sys
import torch

print("Python executable:", sys.executable)
print("PyTorch version:  ", torch.__version__)
print("CUDA available:   ", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "CUDA is not available. In Google Colab, select Runtime > Change runtime type > GPU, "
    "then rerun this cell."
)

device = torch.device("cuda")
print("Active GPU:       ", torch.cuda.get_device_name(0))


In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import drive, files
from kaggle.api.kaggle_api_extended import KaggleApi

# Mount Google Drive. Checkpoints/results persist here between Colab sessions.
drive.mount('/content/drive')

# Keep the dataset on Colab's local disk for faster image loading; save model weights to Drive.
BASE_DIR = Path('/content/drive/MyDrive/Breast_Cancer_Research')
DATA_DIR = Path('/content/breakhis_dataset')
SAVE_DIR = BASE_DIR / 'breakhis_savedmodels' / 'breakhis_80_10_10'
BASE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print('Dataset directory (Colab local disk):', DATA_DIR)
print('80:10:10 weights/results directory (Google Drive):', SAVE_DIR)

# Kaggle credentials: use /content/drive/MyDrive/kaggle.json, or upload it once when prompted.
# Do not paste Kaggle tokens directly into notebook code. If an old token was exposed, revoke it.
kaggle_json = Path('/content/drive/MyDrive/kaggle.json')
if not kaggle_json.exists():
    print('Upload your Kaggle API kaggle.json file (Kaggle > Account > Create New Token).')
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('Please upload a file named kaggle.json.')
    kaggle_json.write_bytes(uploaded['kaggle.json'])
os.environ['KAGGLE_CONFIG_DIR'] = str(kaggle_json.parent)
kaggle_json.chmod(0o600)

api = KaggleApi()
api.authenticate()
print('Kaggle authenticated successfully.')

# Target dataset requested: waseemalastal/breakhis-breast-cancer-histopathological-dataset
dataset_slug = 'waseemalastal/breakhis-breast-cancer-histopathological-dataset'
existing_images = [p for p in DATA_DIR.rglob('*') if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}]
if len(existing_images) < 1000:
    print(f'Downloading {dataset_slug} to Colab local storage...')
    api.dataset_download_files(dataset_slug, path=str(DATA_DIR), unzip=True)
    print('Download and extraction completed.')
else:
    print(f'BreaKHis dataset already detected locally ({len(existing_images)} images found).')


In [ ]:
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

def parse_breakhis_label(path: Path):
    name = path.name.lower()
    parts = [p.lower() for p in path.parts]
    joined = "/".join(parts)
    
    if any("malignant" in p for p in parts) or name.startswith("sob_m_") or "_m_" in name:
        return 1
    if any("benign" in p for p in parts) or name.startswith("sob_b_") or "_b_" in name:
        return 0
    return None

def extract_patient_id(path: Path):
    stem = path.stem
    # Standard BreaKHis naming convention: SOB_<M/B>_<Type>-<Year>-<SlideID>-<Mag>-<Seq>
    match = re.match(r"^(SOB_[A-Za-z]_[A-Za-z]+-\d+-[^-]+)-\d+-\\d+$", stem)
    if match:
        return match.group(1)
    parts = stem.split("-")
    if len(parts) >= 3 and parts[-1].isdigit() and parts[-2].isdigit():
        return "-".join(parts[:-2])
    return path.parent.name

rows = []
for p in DATA_DIR.rglob("*"):
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
        label = parse_breakhis_label(p)
        if label is not None:
            rows.append({
                "path": str(p),
                "label": int(label),
                "patient_id": extract_patient_id(p),
                "filename": p.name
            })

df = pd.DataFrame(rows)
df = df.drop_duplicates(subset=["filename"]).reset_index(drop=True)
print(f"Total Images Found: {len(df)}")
print(f"Unique Patients:    {df['patient_id'].nunique()}")
print(f"Class Distribution: Benign(0)={sum(df['label']==0)} | Malignant(1)={sum(df['label']==1)}")

# Patient-Level Stratified Split (80% Train, 10% Val, 10% Test)
# Each patient is assigned to exactly one partition to avoid patient leakage.
patient_df = df.drop_duplicates(subset=["patient_id"]).reset_index(drop=True)
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
splits = list(sgkf.split(patient_df, patient_df["label"], groups=patient_df["patient_id"]))

# Two folds (~10% of patients each) are assigned to validation and test; the other eight to training.
val_patient_idx = splits[0][1]
test_patient_idx = splits[1][1]
val_patients = set(patient_df.iloc[val_patient_idx]["patient_id"])
test_patients = set(patient_df.iloc[test_patient_idx]["patient_id"])
train_patients = set(patient_df["patient_id"]) - val_patients - test_patients

train_df = df[df["patient_id"].isin(train_patients)].reset_index(drop=True)
val_df = df[df["patient_id"].isin(val_patients)].reset_index(drop=True)
test_df = df[df["patient_id"].isin(test_patients)].reset_index(drop=True)

# Sanity checks: no patient can appear in more than one partition.
assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)

print(f"\nTrain Set: {len(train_df)} images ({train_df['patient_id'].nunique()} patients; {train_df['patient_id'].nunique()/patient_df.shape[0]:.1%} of patients)")
print(f"Val Set:   {len(val_df)} images ({val_df['patient_id'].nunique()} patients; {val_df['patient_id'].nunique()/patient_df.shape[0]:.1%} of patients)")
print(f"Test Set:  {len(test_df)} images ({test_df['patient_id'].nunique()} patients; {test_df['patient_id'].nunique()/patient_df.shape[0]:.1%} of patients)")
print("Image proportions: ", f"train={len(train_df)/len(df):.1%}, val={len(val_df)/len(df):.1%}, test={len(test_df)/len(df):.1%}")


In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMG_SIZE = 224
BATCH_SIZE = 16  # Conservative for the four transformer backbones; increase to 32 only if GPU memory allows.

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class BreaKHisDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(row["label"], dtype=torch.long)

# Colab GPU-friendly loading. Dataset images are on /content, not mounted Drive.
NUM_WORKERS = 2
LOADER_KWARGS = dict(num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=(NUM_WORKERS > 0))
train_loader = DataLoader(BreaKHisDataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True, **LOADER_KWARGS)
val_loader   = DataLoader(BreaKHisDataset(val_df, eval_transform),   batch_size=BATCH_SIZE, shuffle=False, **LOADER_KWARGS)
test_loader  = DataLoader(BreaKHisDataset(test_df, eval_transform),  batch_size=BATCH_SIZE, shuffle=False, **LOADER_KWARGS)

# Compute inverse class frequencies to counteract class imbalance
class_counts = train_df["label"].value_counts().sort_index().values
weights = torch.tensor([sum(class_counts) / (len(class_counts) * c) for c in class_counts], dtype=torch.float).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

In [ ]:
import timm
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

MODEL_CONFIG = {
    "vit": "vit_base_patch16_224",
    "swin": "swin_tiny_patch4_window7_224",
    "deit": "deit_base_patch16_224",
    "beit": "beit_base_patch16_224"
}

EPOCHS = 30
LR = 2e-5
print(f"Checkpoints will be saved to Google Drive: {SAVE_DIR}")

for model_name, model_arch in MODEL_CONFIG.items():
    print(f"\n{'='*70}\nTRAINING {model_name.upper()} ({model_arch})\n{'='*70}")
    
    model = timm.create_model(model_arch, pretrained=True, num_classes=2).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)
    scaler = torch.amp.GradScaler("cuda")
    
    best_val_f1 = 0.0
    save_path = SAVE_DIR / f"{model_name}_best.pth"
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss, total_train = 0.0, 0
        
       
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]", leave=False, dynamic_ncols=True):
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad(set_to_none=True)
    
            with torch.amp.autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)
        
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
    
            running_loss += loss.item() * images.size(0)
            total_train += images.size(0)
            
        scheduler.step()
        train_loss = running_loss / total_train
        
        # Validation
        model.eval()
        v_preds, v_targets = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device, non_blocking=True)
                with torch.amp.autocast("cuda"):
                    outputs = model(images)
                preds = torch.argmax(outputs, dim=1).cpu().tolist()
                v_preds.extend(preds)
                v_targets.extend(labels.tolist())
                
        val_acc = accuracy_score(v_targets, v_preds)
        val_f1 = f1_score(v_targets, v_preds, zero_division=0)
        
        print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), save_path)  # Persistent Google Drive checkpoint: <model>_best.pth
            print(f"  --> Saved new best checkpoint: {save_path.name} (F1: {best_val_f1:.4f})")
            
    # Final Evaluation on Held-Out Test Set
    print(f"\n--- Testing Best Checkpoint for {model_name.upper()} ---")
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval()
    
    t_preds, t_probs, t_targets = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast("cuda"):
                outputs = model(images)
            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().tolist()
            preds = torch.argmax(outputs, dim=1).cpu().tolist()
            t_probs.extend(probs)
            t_preds.extend(preds)
            t_targets.extend(labels.tolist())
            
    print(f"Test Accuracy: {accuracy_score(t_targets, t_preds):.4f}")
    print(f"Test F1 Score: {f1_score(t_targets, t_preds, zero_division=0):.4f}")
    print(f"Test ROC-AUC:  {roc_auc_score(t_targets, t_probs):.4f}")
    print("Confusion Matrix:\n", confusion_matrix(t_targets, t_preds))
    
    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()

In [ ]:
from pathlib import Path

SAVE_DIR = Path(SAVE_DIR)  # 80:10:10 checkpoint folder defined in setup cell
MODELS = ["vit", "swin", "deit", "beit"]

print(f"{'='*60}\nCHECKPOINT INTEGRITY & EXISTENCE AUDIT\n{'='*60}")
missing = []

for m in MODELS:
    ckpt = SAVE_DIR / f"{m}_best.pth"
    if ckpt.exists() and ckpt.stat().st_size > 0:
        mb = ckpt.stat().st_size / (1024 * 1024)
        print(f"[FOUND]     {m.upper():<6} -> {ckpt.name:<18} ({mb:.2f} MB)")
    else:
        print(f"[NOT FOUND] {m.upper():<6} -> Expected: {ckpt.name}")
        missing.append(m)

print(f"{'-'*60}")
assert not missing, f"Missing checkpoints for: {missing}. Resolve before evaluation."
print(f"All {len(MODELS)} model checkpoints are verified and ready for evaluation.")

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)

# 1. Paths and device configuration
SAVE_DIR = Path(SAVE_DIR)  # 80:10:10 checkpoint folder defined in setup cell
SAVE_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Model dictionary mapping keys to timm architecture names
MODEL_CONFIG = {
    "vit": "vit_base_patch16_224",
    "swin": "swin_tiny_patch4_window7_224",
    "deit": "deit_base_patch16_224",
    "beit": "beit_base_patch16_224",
}

BINARY_CLASSES = ["Benign", "Malignant"]

# 3. Iterate through each model checkpoint
for model_name, model_arch in MODEL_CONFIG.items():
    ckpt_path = SAVE_DIR / f"{model_name}_best.pth"

    if not ckpt_path.exists():
        print(f"\nSkipping {model_name.upper()}: Checkpoint not found at {ckpt_path}")
        continue

    print(f"\n{'='*60}")
    print(f"Generating Evaluation Plots for: {model_name.upper()}")
    print(f"{'='*60}")

    # Instantiate model architecture with binary head
    model = timm.create_model(model_arch, pretrained=False, num_classes=2)

    # Handle both direct state_dict and wrapped checkpoint dictionaries
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        model.load_state_dict(checkpoint["state_dict"])
    else:
        model.load_state_dict(checkpoint)

    model = model.to(device)
    model.eval()

    all_targets = []
    all_preds = []
    all_probs = []

    # Inference loop
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(images)

            # Extract malignant probability (index 1) and predicted class
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)

            all_targets.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())

    y_true = np.array(all_targets)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)

    # -------------------------------------------------------------
    # A. Confusion Matrix (Purple Palette)
    # -------------------------------------------------------------
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=BINARY_CLASSES
    )
    # "Purples" provides a purple sequential gradient
    disp.plot(cmap="Purples", ax=ax, values_format="d", colorbar=True)

    ax.set_title(
        f"{model_name.upper()} - Binary Confusion Matrix",
        fontsize=13,
        fontweight="bold",
        pad=12,
    )
    plt.tight_layout()

    cm_save_path = SAVE_DIR / f"{model_name}_confusion_matrix.png"
    plt.savefig(cm_save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved Confusion Matrix: {cm_save_path.name}")

    # -------------------------------------------------------------
    # B. ROC-AUC Curve
    # -------------------------------------------------------------
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = roc_auc_score(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(6, 5))
    # Line plotted in deep purple (#6a0dad)
    ax.plot(
        fpr,
        tpr,
        color="#6a0dad",
        lw=2.5,
        label=f"{model_name.upper()} (AUC = {auc_val:.4f})",
    )
    ax.plot(
        [0, 1],
        [0, 1],
        color="grey",
        lw=1.5,
        linestyle="--",
        label="Chance (AUC = 0.5000)",
    )

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
    ax.set_ylabel("True Positive Rate (Sensitivity)", fontsize=11)
    ax.set_title(
        f"{model_name.upper()} - ROC Curve",
        fontsize=13,
        fontweight="bold",
        pad=12,
    )
    ax.legend(loc="lower right", fontsize=10)
    ax.grid(True, linestyle=":", alpha=0.6)
    plt.tight_layout()

    roc_save_path = SAVE_DIR / f"{model_name}_roc_auc.png"
    plt.savefig(roc_save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved ROC Curve:        {roc_save_path.name}")

    # Free memory between models
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nAll model plots have been generated and saved.")

In [ ]:
# ATTENTION MAPS FOR BEST PERFORMED MODEL SWIN

In [ ]:


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import cv2
import timm
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image


# ============================================================
# 2. PATHS
# ============================================================

SAVE_DIR = Path(SAVE_DIR)  # 80:10:10 checkpoint folder on Google Drive

SWIN_WEIGHTS = SAVE_DIR / "swin_best.pth"

OUTPUT_DIR = SAVE_DIR / "swin_attention"

# Separate folders
ORIGINAL_DIR = OUTPUT_DIR / "original"
HEATMAP_DIR = OUTPUT_DIR / "heatmap"
OVERLAY_DIR = OUTPUT_DIR / "overlay"

BENIGN_DIR = OUTPUT_DIR / "Benign"
MALIGNANT_DIR = OUTPUT_DIR / "Malignant"

# Create folders
for folder in [
    OUTPUT_DIR,
    ORIGINAL_DIR,
    HEATMAP_DIR,
    OVERLAY_DIR,
    BENIGN_DIR,
    MALIGNANT_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 3. SETTINGS
# ============================================================

N_BENIGN = 50
N_MALIGNANT = 50

TOTAL_IMAGES = N_BENIGN + N_MALIGNANT

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("SWIN TRANSFORMER XAI / ACTIVATION MAP GENERATION")
print("=" * 70)

print("Device:", device)

print("Weights:")
print(SWIN_WEIGHTS)

print("\nOutput:")
print(OUTPUT_DIR)


# ============================================================
# 4. CHECK WEIGHTS
# ============================================================

if not SWIN_WEIGHTS.exists():

    raise FileNotFoundError(
        f"Swin weights not found:\n{SWIN_WEIGHTS}"
    )


# ============================================================
# 5. LOAD SWIN MODEL
# ============================================================

print("\nLoading Swin Transformer...")

model = timm.create_model(
    "swin_tiny_patch4_window7_224",
    pretrained=False,
    num_classes=2
)


checkpoint = torch.load(
    SWIN_WEIGHTS,
    map_location=device,
    weights_only=False
)


# ------------------------------------------------------------
# Handle different checkpoint formats
# ------------------------------------------------------------

if isinstance(checkpoint, dict):

    if "state_dict" in checkpoint:

        state_dict = checkpoint["state_dict"]

    elif "model_state_dict" in checkpoint:

        state_dict = checkpoint["model_state_dict"]

    else:

        state_dict = checkpoint

else:

    state_dict = checkpoint


# Remove possible "module." prefix
clean_state_dict = {}

for key, value in state_dict.items():

    if key.startswith("module."):

        key = key.replace(
            "module.",
            "",
            1
        )

    clean_state_dict[key] = value


model.load_state_dict(
    clean_state_dict,
    strict=True
)

model = model.to(device)

model.eval()

print("Swin model loaded successfully.")


# ============================================================
# 6. CLASS MAPPING
# ============================================================

CLASS_NAMES = {
    0: "Benign",
    1: "Malignant"
}


# ============================================================
# 7. SWIN FEATURE HOOK
# ============================================================

class SwinFeatureHook:

    def __init__(self, module):

        self.feature_map = None

        self.hook = module.register_forward_hook(
            self.hook_fn
        )

    def hook_fn(
        self,
        module,
        inputs,
        output
    ):

        self.feature_map = output.detach()

    def close(self):

        self.hook.remove()


# ------------------------------------------------------------
# Use the final Swin block
# ------------------------------------------------------------

target_layer = (
    model.layers[-1]
    .blocks[-1]
    .norm2
)


hook = SwinFeatureHook(
    target_layer
)

print(
    "\nHook attached to:",
    target_layer
)


# ============================================================
# 8. CHECK TEST DATASET
# ============================================================

print("\nTest dataset size:")

print(
    len(test_loader.dataset)
)


# ============================================================
# 9. FIND BENIGN AND MALIGNANT TEST INDICES
# ============================================================

print("\nFinding test images...")


# We use the dataframe already associated
# with your BreaKHis test dataset.

test_df = test_loader.dataset.df


# ------------------------------------------------------------
# Find class column automatically
# ------------------------------------------------------------

possible_label_columns = [
    "label",
    "target",
    "class",
    "class_id",
    "label_id"
]


label_column = None

for col in possible_label_columns:

    if col in test_df.columns:

        label_column = col
        break


# ------------------------------------------------------------
# If no label column exists, use dataset access
# ------------------------------------------------------------

if label_column is not None:

    benign_indices = [
        i for i, value in
        enumerate(test_df[label_column])
        if int(value) == 0
    ]

    malignant_indices = [
        i for i, value in
        enumerate(test_df[label_column])
        if int(value) == 1
    ]

else:

    benign_indices = []
    malignant_indices = []

    for i in range(
        len(test_loader.dataset)
    ):

        _, label = test_loader.dataset[i]

        if int(label) == 0:

            benign_indices.append(i)

        elif int(label) == 1:

            malignant_indices.append(i)


print(
    "Available Benign test images:",
    len(benign_indices)
)

print(
    "Available Malignant test images:",
    len(malignant_indices)
)


# ============================================================
# 10. SELECT EXACTLY 50 + 50

if len(benign_indices) < N_BENIGN:

    raise ValueError(
        f"Only {len(benign_indices)} "
        f"Benign images available. "
        f"Need {N_BENIGN}."
    )


if len(malignant_indices) < N_MALIGNANT:

    raise ValueError(
        f"Only {len(malignant_indices)} "
        f"Malignant images available. "
        f"Need {N_MALIGNANT}."
    )


# ------------------------------------------------------------
# Deterministic selection
# ------------------------------------------------------------

selected_benign = benign_indices[
    :N_BENIGN
]

selected_malignant = malignant_indices[
    :N_MALIGNANT
]


selected_indices = (
    selected_benign +
    selected_malignant
)


print(
    "\nSelected:",
    len(selected_indices)
)

print(
    "Benign:",
    len(selected_benign)
)

print(
    "Malignant:",
    len(selected_malignant)
)

# 11. ACTIVATION MAP FUNCTION


def create_activation_map(
    feature_map
):

    """
    Convert final Swin feature representation
    into a 2D spatial activation map.

    This is a feature activation map / attention-style
    visualization, not raw Swin attention weights.
    """

    feats = feature_map


    # --------------------------------------------------------
    # Case 1: [B, L, C]
    # --------------------------------------------------------

    if feats.dim() == 3:

        B, L, C = feats.shape

        H = int(
            np.sqrt(L)
        )

        W = H

        if H * W != L:

            raise ValueError(
                f"Cannot convert {L} tokens "
                f"into square spatial map."
            )

        feats = feats.reshape(
            B,
            H,
            W,
            C
        )


    # --------------------------------------------------------
    # Case 2: [B, H, W, C]
    # --------------------------------------------------------

    elif feats.dim() == 4:

        pass


    else:

        raise ValueError(
            f"Unexpected Swin feature shape: "
            f"{feats.shape}"
        )


    # --------------------------------------------------------
    # Average channels
    # --------------------------------------------------------

    activation = (
        feats[0]
        .mean(dim=-1)
    )


    activation = (
        activation
        .cpu()
        .numpy()
    )


    # --------------------------------------------------------
    # Normalize 0-1
    # --------------------------------------------------------

    activation = (
        activation -
        activation.min()
    )

    activation = (
        activation /
        (activation.max() + 1e-8)
    )


    return activation

# 12. GENERATE 100 MAPS

metadata = []


print("\n")
print("=" * 70)
print("GENERATING ACTIVATION MAPS")
print("=" * 70)


for counter, dataset_idx in enumerate(
    selected_indices,
    start=1
):


    # ========================================================
    # LOAD IMAGE
    # ========================================================

    image_tensor, true_label = (
        test_loader.dataset[
            dataset_idx
        ]
    )


    true_label = int(
        true_label
    )

    true_class = CLASS_NAMES[
        true_label
    ]


    # ========================================================
    # ORIGINAL IMAGE PATH
    # ========================================================

    row = test_df.iloc[
        dataset_idx
    ]


    # --------------------------------------------------------
    # Find path column
    # --------------------------------------------------------

    path_column = None

    for col in [
        "path",
        "image_path",
        "filepath",
        "file_path",
        "image"
    ]:

        if col in test_df.columns:

            path_column = col
            break


    if path_column is None:

        raise ValueError(
            "Could not find image path column "
            "in test dataframe."
        )


    image_path = Path(
        str(row[path_column])
    )


    # ========================================================
    # MODEL INPUT
    # ========================================================

    image_input = (
        image_tensor
        .unsqueeze(0)
        .to(device)
    )


    # ========================================================
    # FORWARD PASS
    # ========================================================

    with torch.no_grad():

        outputs = model(
            image_input
        )


    # ========================================================
    # PREDICTION
    # ========================================================

    probabilities = torch.softmax(
        outputs,
        dim=1
    )


    confidence, prediction = (
        probabilities.max(dim=1)
    )


    prediction = int(
        prediction.item()
    )

    confidence = float(
        confidence.item()
    )


    predicted_class = (
        CLASS_NAMES[
            prediction
        ]
    )


    correct = (
        true_label ==
        prediction
    )


    # ========================================================
    # FEATURE MAP
    # ========================================================

    feature_map = hook.feature_map


    activation = (
        create_activation_map(
            feature_map
        )
    )


    # ========================================================
    # LOAD ORIGINAL IMAGE
    # ========================================================

    original = Image.open(
        image_path
    ).convert("RGB")


    original = original.resize(
        (224, 224)
    )


    original_np = np.array(
        original
    )


    # ========================================================
    # RESIZE ACTIVATION
    # ========================================================

    activation_uint8 = (
        activation * 255
    ).astype(
        np.uint8
    )


    activation_resized = cv2.resize(
        activation_uint8,
        (224, 224),
        interpolation=cv2.INTER_LINEAR
    )


    # ========================================================
    # CREATE HEATMAP
    # ========================================================

    heatmap = cv2.applyColorMap(
        activation_resized,
        cv2.COLORMAP_JET
    )


    heatmap = cv2.cvtColor(
        heatmap,
        cv2.COLOR_BGR2RGB
    )


    # ========================================================
    # CREATE OVERLAY
    # ========================================================

    overlay = cv2.addWeighted(
        original_np,
        0.60,
        heatmap,
        0.40,
        0
    )


    # ========================================================
    # FILE NAME
    # ========================================================

    stem = image_path.stem

    base_name = (
        f"{counter:03d}_"
        f"{true_class}_"
        f"{stem}_"
        f"pred_{predicted_class}_"
        f"conf_{confidence*100:.2f}"
    )


    # ========================================================
    # SAVE ORIGINAL
    # ========================================================

    original_save_path = (
        ORIGINAL_DIR /
        f"{base_name}.png"
    )


    Image.fromarray(
        original_np
    ).save(
        original_save_path
    )


    # ========================================================
    # SAVE HEATMAP
    # ========================================================

    heatmap_save_path = (
        HEATMAP_DIR /
        f"{base_name}.png"
    )


    Image.fromarray(
        heatmap
    ).save(
        heatmap_save_path
    )


    # ========================================================
    # SAVE OVERLAY
    # ========================================================

    overlay_save_path = (
        OVERLAY_DIR /
        f"{base_name}.png"
    )


    Image.fromarray(
        overlay
    ).save(
        overlay_save_path
    )


    # ========================================================
    # SAVE CLASS COPY
    # ========================================================

    class_dir = (
        BENIGN_DIR
        if true_class == "Benign"
        else MALIGNANT_DIR
    )


    class_save_path = (
        class_dir /
        f"{base_name}.png"
    )


    Image.fromarray(
        overlay
    ).save(
        class_save_path
    )


    # ========================================================
    # METADATA
    # ========================================================

    metadata.append({

        "Map_ID":
            counter,

        "Dataset_Index":
            dataset_idx,

        "Original_Image":
            str(image_path),

        "True_Class":
            true_class,

        "Predicted_Class":
            predicted_class,

        "Confidence":
            confidence,

        "Confidence_Percent":
            confidence * 100,

        "Correct":
            correct,

        "Original_Saved":
            str(original_save_path),

        "Heatmap_Saved":
            str(heatmap_save_path),

        "Overlay_Saved":
            str(overlay_save_path),

        "Class_Map_Saved":
            str(class_save_path)
    })


    # ========================================================
    # PROGRESS
    # ========================================================

    print(
        f"[{counter:03d}/100] "
        f"{true_class:<10} | "
        f"Pred: {predicted_class:<10} | "
        f"Confidence: {confidence*100:6.2f}% | "
        f"{'CORRECT' if correct else 'WRONG'}"
    )

# 13. CLOSE HOOK
hook.close()

# 14. SAVE METADATA CSV

metadata_df = pd.DataFrame(
    metadata
)


METADATA_PATH = (
    OUTPUT_DIR /
    "swin_100_attention_metadata.csv"
)


metadata_df.to_csv(
    METADATA_PATH,
    index=False,
    encoding="utf-8"
)

# 15. CREATE CONFIDENCE SUMMARY
summary = (
    metadata_df
    .groupby("True_Class")
    .agg(
        Number_of_Images=(
            "Map_ID",
            "count"
        ),

        Mean_Confidence=(
            "Confidence_Percent",
            "mean"
        ),

        Min_Confidence=(
            "Confidence_Percent",
            "min"
        ),

        Max_Confidence=(
            "Confidence_Percent",
            "max"
        ),

        Correct_Predictions=(
            "Correct",
            "sum"
        )
    )
    .reset_index()
)


SUMMARY_PATH = (
    OUTPUT_DIR /
    "swin_100_attention_summary.csv"
)


summary.to_csv(
    SUMMARY_PATH,
    index=False
)


# ============================================================
# 16. FINAL INFORMATION
# ============================================================

print("\n")
print("=" * 70)
print("100 SWIN ACTIVATION MAPS GENERATED")
print("=" * 70)

print(
    "\nTotal maps:",
    len(metadata_df)
)

print(
    "Benign:",
    len(
        metadata_df[
            metadata_df["True_Class"]
            == "Benign"
        ]
    )
)

print(
    "Malignant:",
    len(
        metadata_df[
            metadata_df["True_Class"]
            == "Malignant"
        ]
    )
)

print(
    "\nMetadata:",
    METADATA_PATH
)

print(
    "\nSummary:",
    SUMMARY_PATH
)

print(
    "\nAll outputs:",
    OUTPUT_DIR
)

print("\nFolder structure:")
print(
    "  original/"
)
print(
    "  heatmap/"
)
print(
    "  overlay/"
)
print(
    "  Benign/"
)
print(
    "  Malignant/"
)

In [ ]:
!pip -q install ollama


**Optional VLM step (not required for training the four classifiers):** the original `ollama run gemma4:12b` command is a terminal command, not Python. In Colab, Ollama must be installed and its server started before running the Gemma analysis cells. This notebook’s required GPU training and Drive checkpoint saving are complete without running the optional Gemma cells.


In [ ]:
!pip install openpyxl

In [ ]:
# ============================================================
# FIX GROUP-LEVEL GEMMA OUTPUT
# Parse existing Gemma responses and create final table
# ============================================================

import re
import json
import pandas as pd
from pathlib import Path


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = Path(SAVE_DIR) / "swin_attention" / "final_25_heatmaps" / "gemma_group_results"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# FEATURES
# ============================================================

FEATURES = [
    "Texture Variation",
    "Shape Irregularity",
    "Intensity Variation",
    "Tissue Boundaries",
    "Localized Structures",
    "Distributed Structures",
    "Heterogeneity",
    "Overall Pattern",
    "Swin XAI Interpretation"
]


# ============================================================
# ROBUST TEXT PARSER
# ============================================================

def parse_gemma_text(text):

    results = {}

    # Normalize text
    text = text.replace("\r", "\n")

    # Remove markdown bold
    text = re.sub(r"\*\*", "", text)

    # --------------------------------------------------------
    # Multiple possible heading names
    # --------------------------------------------------------

    aliases = {

        "Texture Variation": [
            "TEXTURE VARIATION",
            "TEXTURE",
            "TEXTURAL VARIATION"
        ],

        "Shape Irregularity": [
            "SHAPE IRREGULARITY",
            "SHAPE"
        ],

        "Intensity Variation": [
            "INTENSITY VARIATION",
            "INTENSITY"
        ],

        "Tissue Boundaries": [
            "TISSUE BOUNDARIES",
            "TISSUE BOUNDARY",
            "BOUNDARIES"
        ],

        "Localized Structures": [
            "LOCALIZED STRUCTURES",
            "LOCALIZED STRUCTURE",
            "LOCAL STRUCTURES"
        ],

        "Distributed Structures": [
            "DISTRIBUTED STRUCTURES",
            "DISTRIBUTED STRUCTURE"
        ],

        "Heterogeneity": [
            "HETEROGENEITY",
            "SPATIAL HETEROGENEITY"
        ],

        "Overall Pattern": [
            "OVERALL PATTERN",
            "OVERALL IMAGE SUMMARY",
            "OVERALL SUMMARY",
            "SUMMARY"
        ],

        "Swin XAI Interpretation": [
            "SWIN XAI INTERPRETATION",
            "SWIN XAI",
            "XAI INTERPRETATION",
            "SWIN INTERPRETATION"
        ]
    }


    # --------------------------------------------------------
    # Search each section
    # --------------------------------------------------------

    for feature, possible_names in aliases.items():

        found = None

        for name in possible_names:

            # Match heading followed by colon
            pattern = (
                rf"(?:^|\n)\s*"
                rf"{re.escape(name)}"
                rf"\s*:?\s*"
                rf"(.*?)(?=\n\s*[A-Z][A-Z0-9 \-/]+"
                rf"\s*:|\Z)"
            )

            match = re.search(
                pattern,
                text,
                flags=re.IGNORECASE | re.DOTALL
            )

            if match:

                found = match.group(1).strip()

                if found:
                    break


        if found:

            # Clean excessive whitespace
            found = re.sub(
                r"\s+",
                " ",
                found
            ).strip()

            results[feature] = found

        else:

            results[feature] = None


    return results


# ============================================================
# DEBUG: SHOW RAW GEMMA RESPONSES
# ============================================================

print("=" * 80)
print("CHECKING GEMMA RESPONSES")
print("=" * 80)

print("\nBENIGN RAW RESPONSE:")
print("-" * 80)
print(benign_response)

print("\n\nMALIGNANT RAW RESPONSE:")
print("-" * 80)
print(malignant_response)


# ============================================================
# PARSE
# ============================================================

benign_results = parse_gemma_text(
    benign_response
)

malignant_results = parse_gemma_text(
    malignant_response
)


# ============================================================
# CHECK PARSING
# ============================================================

print("\n\n")
print("=" * 80)
print("PARSING CHECK")
print("=" * 80)

print("\nBENIGN:")

for feature in FEATURES:

    value = benign_results.get(feature)

    if value:
        print(f"✓ {feature}")
    else:
        print(f"✗ {feature}")


print("\nMALIGNANT:")

for feature in FEATURES:

    value = malignant_results.get(feature)

    if value:
        print(f"✓ {feature}")
    else:
        print(f"✗ {feature}")


# ============================================================
# CREATE FINAL TABLE
# ============================================================

rows = []

for feature in FEATURES:

    rows.append({

        "Feature": feature,

        "Benign Explanation":
            benign_results.get(
                feature
            ) or "Not provided",

        "Malignant Explanation":
            malignant_results.get(
                feature
            ) or "Not provided"

    })


final_df = pd.DataFrame(rows)


# ============================================================
# PRINT FINAL TABLE
# ============================================================

print("\n\n")

print("=" * 160)
print("FINAL BENIGN vs MALIGNANT GROUP-LEVEL SWIN XAI TABLE")
print("=" * 160)

for _, row in final_df.iterrows():

    print("\n")
    print(
        f"FEATURE: {row['Feature']}"
    )

    print("-" * 140)

    print(
        "BENIGN:"
    )

    print(
        row["Benign Explanation"]
    )

    print()

    print(
        "MALIGNANT:"
    )

    print(
        row["Malignant Explanation"]
    )

    print("-" * 140)


# ============================================================
# SAVE CSV
# ============================================================

csv_path = (
    OUTPUT_DIR /
    "final_benign_malignant_comparison.csv"
)

final_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# SAVE MARKDOWN
# ============================================================

markdown_path = (
    OUTPUT_DIR /
    "final_benign_malignant_comparison.md"
)

with open(
    markdown_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "# Benign vs Malignant Swin XAI Comparison\n\n"
    )

    f.write(
        final_df.to_markdown(
            index=False
        )
    )


# ============================================================
# SAVE RAW RESPONSES
# ============================================================

with open(
    OUTPUT_DIR /
    "benign_raw_gemma_response.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        benign_response
    )


with open(
    OUTPUT_DIR /
    "malignant_raw_gemma_response.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        malignant_response
    )


# ============================================================
# EXCEL
# ============================================================

excel_path = (
    OUTPUT_DIR /
    "final_benign_malignant_comparison.xlsx"
)

try:

    import openpyxl

    final_df.to_excel(
        excel_path,
        index=False,
        sheet_name="Swin XAI Comparison"
    )

    print(
        f"\n✓ Excel saved:\n{excel_path}"
    )

except ImportError:

    print(
        "\n⚠ openpyxl is not installed."
    )

    print(
        "CSV and Markdown were saved successfully."
    )

    print(
        "\nTo enable Excel, run:"
    )

    print(
        "!pip install openpyxl"
    )


# ============================================================
# FINAL
# ============================================================

print("\n")
print("=" * 80)
print("FILES SAVED")
print("=" * 80)

print(
    f"\nCSV:\n{csv_path}"
)

print(
    f"\nMarkdown:\n{markdown_path}"
)

print(
    "\nRaw Benign response:"
)

print(
    OUTPUT_DIR /
    "benign_raw_gemma_response.txt"
)

print(
    "\nRaw Malignant response:"
)

print(
    OUTPUT_DIR /
    "malignant_raw_gemma_response.txt"
)

print("\nDONE.")

In [ ]:
# ============================================================
# GEMMA 4:12B
# GROUP-LEVEL SWIN XAI ANALYSIS
#
# 25 BENIGN HEATMAPS  -> ONE CONSOLIDATED BENIGN ANALYSIS
# 25 MALIGNANT HEATMAPS -> ONE CONSOLIDATED MALIGNANT ANALYSIS
#
# FINAL OUTPUT:
# Feature | Benign Explanation | Malignant Explanation
# ============================================================

!pip install -q ollama pandas pillow openpyxl

import os
import re
import pandas as pd
from pathlib import Path
from PIL import Image
import ollama


# ============================================================
# 1. PATH
# ============================================================

HEATMAP_DIR = Path(SAVE_DIR) / "swin_attention" / "final_25_heatmaps"

OUTPUT_DIR = HEATMAP_DIR / "gemma_group_results"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. MODEL
# ============================================================

MODEL_NAME = "gemma4:12b"

print("=" * 80)
print("CHECKING GEMMA MODEL")
print("=" * 80)

try:

    models = ollama.list()

    available_models = []

    for model in models.models:

        if hasattr(model, "model"):
            available_models.append(model.model)

        elif isinstance(model, dict):
            available_models.append(
                model.get("name", "")
            )

    if not any(
        MODEL_NAME in x
        for x in available_models
    ):

        print(
            f"{MODEL_NAME} is not installed."
        )

        print(
            f"Run in terminal:\n"
            f"ollama pull {MODEL_NAME}"
        )

        raise RuntimeError(
            f"{MODEL_NAME} not found."
        )

    print(
        f"✓ {MODEL_NAME} is available."
    )

except Exception as e:

    print(e)
    raise


# ============================================================
# 3. FIND HEATMAPS
# ============================================================

extensions = [
    "*.png",
    "*.jpg",
    "*.jpeg",
    "*.PNG",
    "*.JPG",
    "*.JPEG"
]

all_images = []

for ext in extensions:

    all_images.extend(
        HEATMAP_DIR.rglob(ext)
    )


all_images = sorted(
    set(all_images)
)


print("\n" + "=" * 80)
print("HEATMAPS FOUND")
print("=" * 80)

print(
    f"Total images: {len(all_images)}"
)


# ============================================================
# 4. IDENTIFY BENIGN / MALIGNANT
# ============================================================

benign_images = []
malignant_images = []


for image_path in all_images:

    path_lower = str(
        image_path
    ).lower()

    if "benign" in path_lower:

        benign_images.append(
            image_path
        )

    elif "malignant" in path_lower:

        malignant_images.append(
            image_path
        )


print(
    f"Benign heatmaps   : {len(benign_images)}"
)

print(
    f"Malignant heatmaps: {len(malignant_images)}"
)


# ============================================================
# 5. CHECK DATA
# ============================================================

if len(benign_images) == 0:

    raise RuntimeError(
        """
No Benign heatmaps found.

Make sure your filenames or folders contain:
'Benign'
"""
    )


if len(malignant_images) == 0:

    raise RuntimeError(
        """
No Malignant heatmaps found.

Make sure your filenames or folders contain:
'Malignant'
"""
    )


# ============================================================
# 6. EXPECTED SAMPLE COUNT
# ============================================================

print("\n")

if len(benign_images) != 25:

    print(
        f"WARNING: Expected 25 Benign images, "
        f"but found {len(benign_images)}."
    )


if len(malignant_images) != 25:

    print(
        f"WARNING: Expected 25 Malignant images, "
        f"but found {len(malignant_images)}."
    )


# ============================================================
# 7. FEATURE CATEGORIES
# ============================================================

FEATURES = [

    "Texture Variation",

    "Shape Irregularity",

    "Intensity Variation",

    "Tissue Boundaries",

    "Localized Structures",

    "Distributed Structures",

    "Heterogeneity",

    "Overall Pattern",

    "Swin XAI Interpretation"

]


# ============================================================
# 8. CREATE GROUP PROMPT
# ============================================================

def create_group_prompt(class_name, number_images):

    prompt = f"""
You are performing a GROUP-LEVEL explainability analysis of
breast cancer histopathology images using Swin Transformer
activation heatmaps.

You have been provided with {number_images} Swin Transformer
heatmaps belonging to the class:

{class_name}

IMPORTANT:

These are NOT original histopathology images.

They are Swin Transformer feature activation heatmaps.

Your task is NOT to describe each image separately.

Instead, analyze ALL {number_images} heatmaps together and identify
the COMMON and REPEATED activation patterns across the entire
{class_name} group.

The purpose is to produce a consolidated research-level description
of how Swin Transformer activation patterns differ between
Benign and Malignant groups.

Do NOT invent microscopic structures that cannot reasonably be
inferred from the activation maps.

Do NOT diagnose patients.

Do NOT claim that activation patterns alone medically prove
malignancy or benignity.

Focus on the spatial activation patterns.

For the {class_name} group, analyze:

1. Texture Variation

Discuss whether the activation patterns across the group tend
to be relatively uniform or heterogeneous, and describe the
common spatial distribution of activation.

2. Shape Irregularity

Discuss whether high-activation regions commonly appear regular,
irregular, compact, fragmented, asymmetric, or mixed.

3. Intensity Variation

Discuss the common variation between weak and strong activation
regions across the heatmaps.

4. Tissue Boundaries

Discuss whether activation commonly appears near boundaries,
interfaces, or transitions between regions.

5. Localized Structures

Discuss whether the model commonly focuses on localized,
concentrated regions.

6. Distributed Structures

Discuss whether activation is commonly distributed across
multiple spatial regions.

7. Heterogeneity

Describe the overall spatial heterogeneity observed across
the group.

8. Overall Pattern

Give a concise overall description of the common activation
behavior across this class.

9. Swin XAI Interpretation

Explain what these repeated activation patterns suggest about
the visual features that Swin Transformer is using for this
class prediction.

IMPORTANT OUTPUT FORMAT:

Return EXACTLY the following format.

TEXTURE VARIATION:
<group-level explanation>

SHAPE IRREGULARITY:
<group-level explanation>

INTENSITY VARIATION:
<group-level explanation>

TISSUE BOUNDARIES:
<group-level explanation>

LOCALIZED STRUCTURES:
<group-level explanation>

DISTRIBUTED STRUCTURES:
<group-level explanation>

HETEROGENEITY:
<group-level explanation>

OVERALL PATTERN:
<group-level explanation>

SWIN XAI INTERPRETATION:
<group-level explanation>

Each section should contain approximately 2–4 sentences.

Do NOT describe the images one by one.
Produce ONE consolidated explanation for the entire
{class_name} group.
"""

    return prompt


# ============================================================
# 9. SEND ALL IMAGES TO GEMMA
# ============================================================

def analyze_group(
    image_paths,
    class_name
):

    print("\n")
    print("=" * 80)
    print(
        f"ANALYZING ALL {class_name.upper()} HEATMAPS"
    )
    print("=" * 80)

    print(
        f"Number of heatmaps: {len(image_paths)}"
    )

    # --------------------------------------------------------
    # Verify images
    # --------------------------------------------------------

    valid_images = []

    for image_path in image_paths:

        try:

            img = Image.open(
                image_path
            )

            img.verify()

            valid_images.append(
                image_path
            )

        except Exception as e:

            print(
                f"Skipping invalid image: "
                f"{image_path.name}"
            )

            print(e)


    print(
        f"Valid images sent to Gemma: "
        f"{len(valid_images)}"
    )


    # --------------------------------------------------------
    # Create prompt
    # --------------------------------------------------------

    prompt = create_group_prompt(
        class_name,
        len(valid_images)
    )


    # --------------------------------------------------------
    # Create image list
    # --------------------------------------------------------

    image_paths_for_gemma = [
        str(x)
        for x in valid_images
    ]


    print("\nSending ALL heatmaps to Gemma...")
    print(
        "This may take some time because "
        "multiple images are being analyzed together."
    )


    # --------------------------------------------------------
    # GEMMA
    # --------------------------------------------------------

    response = ollama.chat(

        model=MODEL_NAME,

        messages=[

            {
                "role": "user",

                "content": prompt,

                "images": image_paths_for_gemma

            }

        ],

        options={

            "temperature": 0.2

        }

    )


    response_text = (
        response["message"]["content"]
    )


    print("\nGemma analysis completed.")


    return response_text


# ============================================================
# 10. PARSE GEMMA OUTPUT
# ============================================================

def parse_group_response(
    response_text
):

    parsed = {}

    for i, feature in enumerate(FEATURES):

        if i < len(FEATURES) - 1:

            next_feature = FEATURES[i + 1]

            pattern = (

                rf"{re.escape(feature)}\s*:"
                rf"\s*(.*?)"
                rf"(?=\n\s*"
                rf"{re.escape(next_feature)}\s*:)"
            )

        else:

            pattern = (

                rf"{re.escape(feature)}\s*:"
                rf"\s*(.*)"
            )


        match = re.search(
            pattern,
            response_text,
            flags=re.IGNORECASE |
                  re.DOTALL
        )


        if match:

            text_value = (
                match.group(1)
                .strip()
            )

            # Remove unnecessary markdown
            text_value = re.sub(
                r"^\*+",
                "",
                text_value
            ).strip()

            parsed[feature] = (
                text_value
            )

        else:

            parsed[feature] = (
                "Not provided"
            )


    return parsed


# ============================================================
# 11. ANALYZE BENIGN GROUP
# ============================================================

benign_response = analyze_group(
    benign_images,
    "Benign"
)


benign_results = parse_group_response(
    benign_response
)


# ============================================================
# 12. ANALYZE MALIGNANT GROUP
# ============================================================

malignant_response = analyze_group(
    malignant_images,
    "Malignant"
)


malignant_results = parse_group_response(
    malignant_response
)


# ============================================================
# 13. CREATE FINAL COMPARISON TABLE
# ============================================================

final_table = []


for feature in FEATURES:

    final_table.append({

        "Feature": feature,

        "Benign Explanation":
            benign_results.get(
                feature,
                "Not provided"
            ),

        "Malignant Explanation":
            malignant_results.get(
                feature,
                "Not provided"
            )

    })


final_df = pd.DataFrame(
    final_table
)


# ============================================================
# 14. PRINT FINAL TABLE
# ============================================================

print("\n\n")

print("#" * 150)

print(
    "FINAL GROUP-LEVEL SWIN XAI COMPARISON"
)

print("#" * 150)


for _, row in final_df.iterrows():

    print("\n")
    print(
        f"FEATURE: {row['Feature']}"
    )

    print(
        "-" * 120
    )

    print(
        "BENIGN:"
    )

    print(
        row["Benign Explanation"]
    )

    print()

    print(
        "MALIGNANT:"
    )

    print(
        row["Malignant Explanation"]
    )

    print(
        "-" * 120
    )


# ============================================================
# 15. SAVE CSV
# ============================================================

csv_path = (
    OUTPUT_DIR /
    "final_benign_malignant_comparison.csv"
)


final_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 16. SAVE EXCEL
# ============================================================

excel_path = (
    OUTPUT_DIR /
    "final_benign_malignant_comparison.xlsx"
)


final_df.to_excel(
    excel_path,
    index=False
)


# ============================================================
# 17. SAVE MARKDOWN TABLE
# ============================================================

markdown_path = (
    OUTPUT_DIR /
    "final_benign_malignant_comparison.md"
)


with open(
    markdown_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "# Swin Transformer XAI "
        "Benign vs Malignant Comparison\n\n"
    )

    f.write(
        final_df.to_markdown(
            index=False
        )
    )


# ============================================================
# 18. SAVE RAW GEMMA RESPONSES
# ============================================================

with open(
    OUTPUT_DIR /
    "benign_gemma_group_analysis.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(benign_response)


with open(
    OUTPUT_DIR /
    "malignant_gemma_group_analysis.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(malignant_response)


# ============================================================
# 19. FINAL OUTPUT
# ============================================================

print("\n\n")

print("=" * 80)
print("FINAL TABLE CREATED")
print("=" * 80)

print(
    f"\nCSV:\n{csv_path}"
)

print(
    f"\nExcel:\n{excel_path}"
)

print(
    f"\nMarkdown:\n{markdown_path}"
)

print(
    f"\nBenign raw analysis:\n"
    f"{OUTPUT_DIR / 'benign_gemma_group_analysis.txt'}"
)

print(
    f"\nMalignant raw analysis:\n"
    f"{OUTPUT_DIR / 'malignant_gemma_group_analysis.txt'}"
)

print("\nDONE.")